In [ ]:
# Cell 1 — Imports + fixed project parameters
import numpy as np
import scipy.signal as signal
import scipy.io as sio
from scipy.io import wavfile
import matplotlib.pyplot as plt
from IPython.display import Audio, display

faudio = 48_000     # Hz
fs     = 240_000    # Hz
fc     = 110_000    # Hz
up = fs // faudio   # 5

print("fs:", fs, "Hz | faudio:", faudio, "Hz | fc:", fc, "Hz | upsample:", up)
print("Nyquist:", fs/2, "Hz | headroom above fc:", fs/2 - fc, "Hz")


In [ ]:
# Cell 2 — Load WAV, convert to mono float, choose a segment
wav_path = "inna_sunisup.wav"   # <-- change to your file

fs_in, x_in = wavfile.read(wav_path)

# to float [-1, 1]
if x_in.dtype == np.int16:
    x = x_in.astype(np.float64) / 32768.0
elif x_in.dtype == np.int32:
    x = x_in.astype(np.float64) / 2147483648.0
else:
    x = x_in.astype(np.float64)

# stereo -> mono
if x.ndim == 2:
    x = x.mean(axis=1)

print("Input fs:", fs_in, "Hz | duration:", len(x)/fs_in, "s")

# choose segment
t0  = 0.0  # start time (s)
dur = 10.0   # duration (s)

i0 = int(t0 * fs_in)
i1 = int((t0 + dur) * fs_in)
seg = x[i0:i1]

# normalize
seg = seg - np.mean(seg)
seg = seg / (np.max(np.abs(seg)) + 1e-12)

display(Audio(seg, rate=fs_in))


In [ ]:
# Cell 3 — Resample segment to faudio=48 kHz
seg_48k = signal.resample_poly(seg, up=faudio, down=fs_in)
seg_48k = seg_48k - np.mean(seg_48k)
seg_48k = seg_48k / (np.max(np.abs(seg_48k)) + 1e-12)

print("Segment @48k duration:", len(seg_48k)/faudio, "s")
display(Audio(seg_48k, rate=faudio))


In [ ]:
# Cell 4 — Helper: FM modulator + spectrum/alias stress test near Nyquist

def fm_modulate(m_fs, fs, fc, delta_f_hz):
    """
    m_fs: message at sampling rate fs (float, ~[-1,1])
    delta_f_hz: peak deviation when message amplitude is 1
    returns x_fm
    """
    m_fs = np.asarray(m_fs, dtype=np.float64)
    m_fs = m_fs - np.mean(m_fs)
    m_fs = m_fs / (np.max(np.abs(m_fs)) + 1e-12)

    n = np.arange(len(m_fs))
    t = n / fs
    phase = 2*np.pi*fc*t + 2*np.pi*delta_f_hz * np.cumsum(m_fs) / fs
    return np.cos(phase)

def compute_fft(sig, fs):
    sig = np.asarray(sig, dtype=np.float64)
    sig = sig - np.mean(sig)
    N = len(sig)
    w = np.hanning(N)
    X = np.fft.fft(sig * w)
    f = np.fft.fftfreq(N, d=1/fs)
    X = np.fft.fftshift(X)
    f = np.fft.fftshift(f)
    P = (np.abs(X) ** 2)
    return f, P

def plot_spectrum(sig, fs, title, f_lim_hz=None):
    f, P = compute_fft(sig, fs)
    plt.figure(figsize=(6, 5))
    plt.plot(f/1e3, 10*np.log10(P + 1e-30))
    plt.title(title)
    plt.xlabel("Frequency (kHz)")
    plt.ylabel("Power (dB, relative)")
    plt.grid(True, alpha=0.3)
    if f_lim_hz is not None:
        plt.xlim(-f_lim_hz/1e3, f_lim_hz/1e3)
    plt.tight_layout()
    plt.show()

def alias_energy_ratio(sig, fs, f_stop=120_000, guard_hz=0):
    """
    Estimate how much spectral power lies beyond (f_stop-guard_hz) in magnitude.
    With fs=240k, f_stop should be 120k (Nyquist). guard_hz lets you measure
    power near/above a conservative limit (e.g., 118k).
    Returns (ratio, power_out, power_total).
    """
    f, P = compute_fft(sig, fs)
    f_abs = np.abs(f)
    lim = f_stop - guard_hz
    total = np.sum(P)
    out = np.sum(P[f_abs >= lim])
    ratio = out / (total + 1e-30)
    return ratio, out, total


In [ ]:
# Cell 5 — Define candidate parameter sets (Option A vs Option B, plus a safer one)

candidates = [
    {"name": "Option A (fm=8k, Δf=2k)", "audio_bw_hz": 8_000, "delta_f_hz": 2_000},
    {"name": "Option B (fm=5k, Δf=5k)", "audio_bw_hz": 5_000, "delta_f_hz": 5_000},
    {"name": "Safer (fm=5k, Δf=2k)",    "audio_bw_hz": 5_000, "delta_f_hz": 2_000},
]
candidates


In [ ]:
# Cell 6 — Build signals for each candidate, compute alias risk, and preview demod audio

results = []

for c in candidates:
    audio_bw_hz = c["audio_bw_hz"]
    delta_f_hz  = c["delta_f_hz"]

    # 1) Low-pass at faudio
    numtaps = 401
    h_audio = signal.firwin(numtaps, cutoff=audio_bw_hz, fs=faudio, window="hamming")
    a_lp = signal.filtfilt(h_audio, [1.0], seg_48k)
    a_lp = a_lp / (np.max(np.abs(a_lp)) + 1e-12)

    # 2) Upsample to fs
    m_fs = signal.resample_poly(a_lp, up=up, down=1)
    m_fs = m_fs - np.mean(m_fs)
    m_fs = m_fs / (np.max(np.abs(m_fs)) + 1e-12)

    # 3) FM modulate
    x_fm = fm_modulate(m_fs, fs=fs, fc=fc, delta_f_hz=delta_f_hz)

    # 4) Alias energy ratios: at Nyquist and with a 2 kHz guard (118 kHz)
    r_nyq, _, _ = alias_energy_ratio(x_fm, fs, f_stop=120_000, guard_hz=0)
    r_guard, _, _ = alias_energy_ratio(x_fm, fs, f_stop=120_000, guard_hz=2_000)

    # 5) Demod quick (Hilbert phase derivative)
    xa = signal.hilbert(x_fm)
    phi = np.unwrap(np.angle(xa))
    dphi = np.diff(phi)
    f_inst = (fs / (2*np.pi)) * dphi
    f_inst = f_inst - np.mean(f_inst)

    # LPF back to audio_bw_hz
    h_lpf = signal.firwin(401, cutoff=audio_bw_hz, fs=fs, window="hamming")
    mono = signal.filtfilt(h_lpf, [1.0], f_inst)

    # Downsample to 48 kHz
    mono_48 = signal.resample_poly(mono, up=1, down=up)
    mono_48 = mono_48 - np.mean(mono_48)
    mono_48 = mono_48 / (np.max(np.abs(mono_48)) + 1e-12)

    results.append({
        "name": c["name"],
        "audio_bw_hz": audio_bw_hz,
        "delta_f_hz": delta_f_hz,
        "alias_ratio_at_120k": r_nyq,
        "alias_ratio_at_118k": r_guard,
        "x_fm": x_fm,
        "rec_audio_48": mono_48,
        "msg_audio_48": a_lp
    })

# print summary
for r in results:
    print(f"{r['name']}: alias@120k={r['alias_ratio_at_120k']:.3e}, alias@118k={r['alias_ratio_at_118k']:.3e}")


In [ ]:
# Cell 7 — Compare spectra near Nyquist for each candidate
for r in results:
    plot_spectrum(r["x_fm"], fs, title=r["name"] + " | Spectrum near Nyquist", f_lim_hz=120_000)


In [ ]:
# Cell 8 — Listen to recovered audio for each candidate (demod output)
for r in results:
    print(r["name"])
    display(Audio(r["rec_audio_48"], rate=faudio))


In [ ]:
# Cell 9 — Choose best candidate (lowest alias@118k by default), then save fav.mat

best = min(results, key=lambda d: d["alias_ratio_at_118k"])
print("Chosen:", best["name"])
print("audio_bw_hz:", best["audio_bw_hz"], "delta_f_hz:", best["delta_f_hz"])

fav = best["x_fm"].astype(np.float64)

sio.savemat("fav.mat", {
    "fav": fav,
    "fs": fs,
    "fc": fc,
    "faudio": faudio,
    "audio_bw_hz": best["audio_bw_hz"],
    "delta_f_hz": best["delta_f_hz"],
})

print("Saved fav.mat")
